# Autoencoders Accuracy

In [ ]:
import sys
with open("./../../../PATHS.txt") as file:
  paths = file.read().splitlines()
sys.path.extend(paths)

In [ ]:
import os
import json
import numpy as np
import pandas as pd

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D

colors = matplotlib.rcParams["axes.prop_cycle"].by_key()["color"]
markers = Line2D.filled_markers[1:]

In [ ]:
from dd_nm_rom import ops
from dd_nm_rom.postproc.utils import set_style
plt = set_style(plt)

In [ ]:
dim_ranges = {
  "interior": {
    "latent_dim": {
      "start": 12,
      "stop": 37,
      "step": 6
    },
    "row_nonzero": 10
  },
  "port": {
    "latent_dim": {
      "start": 6,
      "stop": 15,
      "step": 2
    },
    "row_nonzero": 8
  }
}
elements = ["interior", "port"]
prefix = "/g/g92/zanardi1/Workspace/Codes/DD-NM-ROM/run/unsteady/dset.2by2/figures/"

In [ ]:
def_styles = {
  "interior": lambda index: dict(
    color=colors[index],
    marker=markers[index],
    markersize=7,
    markerfacecolor=colors[index],
    markeredgecolor=colors[index],
    linestyle=""
  ),
  "port": lambda index: dict(
    color=colors[index],
    linestyle="--"
  )
}

Generate all configurations

In [ ]:
dims, cfgs = {}, []
for element in elements:
  dims[element] = ops.generate_combs([
    np.arange(**dim_ranges[element]["latent_dim"]),
    np.array([dim_ranges[element]["row_nonzero"]])
  ])
  cfgs.append(np.arange(len(dims[element])))
cfgs = ops.generate_combs(cfgs)

Loop over configurations

In [ ]:
def read_stats(filename):
  with open(filename) as file:
    stats = json.load(file)
  error = stats["error"]["srpc"]["mean"]
  speedup = stats["speedup"]["srpc"]["total"]["mean"]
  conv_perc = stats["conv_perc"]["srpc"]
  return error, speedup, conv_perc

In [ ]:
stats = {"conv_perc": [], "error": [], "speedup": []}
for element in elements:
  stats["ld_"+element] = []
for cfg in cfgs:
  # Set tag
  lds, tag = {}, []
  for (e, element) in enumerate(elements):
    ld, rnz = dims[element][cfg[e]]
    lds[element] = ld
    tag.append(f"{element}_ld_{ld}_rnz_{rnz}")
  tag = "_".join(tag)
  # Read stats
  filename = prefix + f"/{tag}/stats_mean.json"
  if os.path.exists(filename):
    error, speedup, conv_perc = read_stats(filename)
    for (element, ld) in lds.items():
      stats["ld_"+element].append(ld)
    stats["error"].append(error)
    stats["speedup"].append(speedup)
    stats["conv_perc"].append(conv_perc)
stats = pd.DataFrame(stats)
stats

Plot Pareto front

In [ ]:
def plot_pareto_fronts(styles, filename):
  fig, ax = plt.subplots()
  for element in ("port", "interior"):
    for ld, style in styles[element].items():
      x = style.pop("x")
      y = style.pop("y")
      l = element[0].upper()
      ax.plot(x, y, **style, label="$\mathcal{%s}-{%d}$" % (l,ld))
  ax.set_xlabel("Speedup")
  ax.set_ylabel("Error")
  ax.invert_xaxis()
  ax.ticklabel_format(axis='y', style='', scilimits=(0,0))
  ax.grid()
  ax.legend(fontsize=15, bbox_to_anchor=(1.04, 0.5), loc="center left", borderaxespad=0)
  plt.savefig(filename, bbox_inches="tight", pad_inches=0.1)
  plt.show()

In [ ]:
pstyles = {}
for element in elements:
  pstyles[element] = {}
  i, x, y = 0, [], []
  for index, istats in stats.iterrows():
    ld = istats["ld_"+element]
    x.append(istats["speedup"])
    y.append(istats["error"])
    if (ld not in pstyles[element]):
      pstyles[element][ld] = def_styles[element](i)
      pstyles[element][ld]["x"] = []
      pstyles[element][ld]["y"] = []
      i += 1
    pstyles[element][ld]["x"].append(istats["speedup"])
    pstyles[element][ld]["y"].append(istats["error"])

In [ ]:
path = prefix + "/global/"
os.makedirs(path, exist_ok=True)
plot_pareto_fronts(pstyles, path+"/pareto.png")